# Network Routing Benchmark

In [1]:
import sys, os
sys.path.append(os.path.expanduser("~/QEVIP/VQA"))

In [2]:
%matplotlib inline
import time
import pandas as pd
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize

from Builders.base import AnsatzBuilder
from AnsatzBenchmarking.Problems.networkTrafficRouting.NetworkTrafficRoutingProblems import NetworkTrafficRoutingProblemSet

from Utilities import cost_func, Estimator
from SLSQP import slsqp

from Builders.MomentumMonteCarlo import momentum_sa_phased as mb
import traceback

from QAdaPrunedMomentumOptimizer import (
    QAdaPrunedMomentumOptimizer,
    OptimizationConfig,
    MomentumTracker
)


In [3]:
class MonteCarloMomentumBuilder(AnsatzBuilder):
    def build(self):
        H = self.hamiltonian
        n_qubits = H.num_qubits
        
        circuit = QuantumCircuit(n_qubits)
        ansatz = QuantumCircuit(n_qubits)

        for i in range(n_qubits):
            pName = f"angle{i}"
            p = Parameter(pName)
            ansatz.rx(p, i)

        paramsList = [1] * n_qubits
        paramsIndex = [i for i in range(n_qubits)]

        # Call the Monte Carlo pipeline
        optimized_ansatz, optimized_params, _ = mb(
            params=paramsList,
            inds=paramsIndex,
            ansatz=ansatz,
            circuit=circuit,
            hamiltonian=H,
            estimator=StatevectorEstimator(),
            beta1=0.9,
            beta2=0.99,
            iters=3,                
            optimization_runs=500
        )

        self.circuit = optimized_ansatz
        self.optimized_params = np.array(optimized_params, dtype=float).tolist()

        return self.circuit

In [4]:
def wrap_cost(cost):
    """Ensure cost_func returns a scalar."""
    if isinstance(cost, np.ndarray):
        return float(cost.item() if cost.size == 1 else cost[0])
    if isinstance(cost, list):
        return float(cost[0] if len(cost) > 0 else 0)
    return float(cost)

def evaluate_builder_routing(builder_class, problem_set, trials=3):
    """Evaluate builder on Network Traffic Routing problems.

    Routing cost is a direct minimization
    """
    problems = problem_set.getProblemSet()
    results = []
    estimator = Estimator()

    for i, (h, exact) in enumerate(problems):
        print(f"\nProblem {i+1}/{len(problems)}: {h.num_qubits} qubits")
        hamiltonian = h  # minimize directly

        for trial in range(trials):
            try:
                builder = builder_class(hamiltonian)

                start = time.time()
                circuit = builder.getCircuit()
                build_time = time.time() - start

                start = time.time()
                params = np.random.random(len(circuit.parameters))
                opt_result = slsqp(
                    func=lambda p: wrap_cost(cost_func(p, circuit, hamiltonian, estimator)),
                    x0=params,
                    maxiter=200,
                )
                opt_time = time.time() - start

                min_cost_energy = opt_result.fun

                ansatz_name = builder_class.__name__.replace('Builder', '')
                if ansatz_name == 'FixedSU2':
                    ansatz_name = 'EfficientSU2'

                result = {
                    'ansatz_type': ansatz_name,
                    'problem_index': i + 1,
                    'trial': trial + 1,
                    'energy': min_cost_energy,
                    'exact_energy': exact if exact is not None else None,
                    'time': build_time + opt_time,
                    'params': len(circuit.parameters),
                    'converged': opt_result.success,
                }
                results.append(result)
                print(f"  Trial {trial+1}: ✓ Energy: {min_cost_energy:.6f}" +
                      (f" (exact: {exact:.6f})" if exact is not None else ""))
            except Exception as e:
                print(f"  Trial {trial+1} failed: {e}")
                traceback.print_exc()

    return results

def evaluate_builder_Qada(builder_class, problem_set, trials=3):
    """Evaluate builder on Network Traffic Routing problems.

    Routing cost is a direct minimization
    """
    problems = problem_set.getProblemSet()
    results = []
    estimator = Estimator()

    for i, (h, exact) in enumerate(problems):
        print(f"\nProblem {i+1}/{len(problems)}: {h.num_qubits} qubits")
        hamiltonian = h  # minimize directly

        for trial in range(trials):
            try:
                builder = builder_class(hamiltonian)
                
                start = time.time()
                #base_circuit = builder.getCircuit()
                base_circuit = QuantumCircuit(h.num_qubits)

                ########################################
                # Create optimizer with aggressive config
                config = OptimizationConfig(
                    max_layers=6,
                    pruning_threshold=0.02,  # More aggressive pruning
                    momentum_beta1=0.95,      # Stronger momentum
                    momentum_beta2=0.99,
                    pruning_window_size=4,    # Shorter window = more responsive
                    patience=4,               # Stop after 4 iterations of no improvement
                    optimization_method="cobyla",  # or "de" for differential evolution
                    max_iterations_per_layer=500
                )

                optimizer = QAdaPrunedMomentumOptimizer(
                    circuit=base_circuit,
                    hamiltonian=h,
                    estimator=estimator,
                    config=config
                )
                
                build_time = time.time() - start

                start = time.time()
                #params = np.random.random(len(circuit.parameters))
                #opt_result = slsqp(
                #    func=lambda p: wrap_cost(cost_func(p, circuit, hamiltonian, estimator)),
                #    x0=params,
                #    maxiter=200,
                #)
                # Run optimization (replaces your LayerOptimizer + manual pruning)
                opt_result = optimizer.optimize(verbose=True)
                ########################################


                opt_time = time.time() - start

                min_cost_energy = opt_result['final_cost']

                ansatz_name = builder_class.__name__.replace('Builder', '')
                if ansatz_name == 'FixedSU2':
                    ansatz_name = 'EfficientSU2'

                result = {
                    'ansatz_type': ansatz_name,
                    'problem_index': i + 1,
                    'trial': trial + 1,
                    'energy': min_cost_energy,
                    'exact_energy': exact if exact is not None else None,
                    'time': build_time + opt_time,
                    'params': opt_result['num_parameters'],
                    'converged': np.isfinite(min_cost_energy),
                }
                results.append(result)
                print(f"  Trial {trial+1}: ✓ Energy: {min_cost_energy:.6f}" +
                      (f" (exact: {exact:.6f})" if exact is not None else ""))
            except Exception as e:
                print(f"  Trial {trial+1} failed: {e}")
                traceback.print_exc()

    return results

In [6]:
from Builders.evolvedOp import EvolvedOperatorBuilder as EfficientSU2Builder
# Load problem set and run benchmarks
print("Loading NetworkTrafficRouting problem set...")
problem_set = NetworkTrafficRoutingProblemSet()
problems = problem_set.getProblemSet()
print(f"Loaded {len(problems)} NetworkTrafficRouting problems\n")

print("Running benchmarks...")
print("="*60)

# Evaluate MomentumBuilder
#print("\n" + "="*60)
#print("MomentumBuilder:")
#print("="*60)
#momentum_results = evaluate_builder_routing(MonteCarloMomentumBuilder, problem_set, trials=3)

# Evaluate EfficientSU2
#print(f"\n{'='*60}")
#print("EfficientSU2:")
#print("="*60)

#su2_results = evaluate_builder_routing(EfficientSU2Builder, problem_set, trials=3)

# Evaluate QAdaprune optimized MomentumBuilder
print("\n" + "="*60)
print("QAdaprune + MomentumBuilder:")
print("="*60)
Qada_results = evaluate_builder_Qada(MonteCarloMomentumBuilder, problem_set, trials=3)

Loading NetworkTrafficRouting problem set...
Loaded 7 NetworkTrafficRouting problems

Running benchmarks...

QAdaprune + MomentumBuilder:

Problem 1/7: 2 qubits
QAdaPrune + Momentum Optimizer

[Layer 0]
  Current parameters: 0
  Total pruned: 0
  Cost: 5.000000
  Active params: 2
  Pruning rate: 0.0%
  Time: 0.133s

[Layer 1]
  Current parameters: 2
  Total pruned: 0
  Cost: 5.000000
  Active params: 4
  Pruning rate: 0.0%
  Time: 0.367s

[Layer 2]
  Current parameters: 4
  Total pruned: 0
  Cost: 5.000000
  Active params: 6
  Pruning rate: 0.0%
  Time: 0.586s

[Layer 3]
  Current parameters: 6
  Total pruned: 0
  Cost: 5.000000
  Active params: 8
  Pruning rate: 0.0%
  Time: 0.636s

[Layer 4]
  Current parameters: 8
  Total pruned: 0
  Cost: 5.000000
  Active params: 10
  Pruning rate: 0.0%
  Time: 0.734s

✓ Converged after 5 layers

Optimization Complete
Final Cost: 5.000000
Layers Added: 5
Final Parameters: 10
Parameters Pruned: 0
Total Time: 2.46s
  Trial 1: ✓ Energy: 5.000000 (exa

# Results

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Add other builders' result lists here to compare them side by side
all_results = {
    "Momentum": momentum_results,
    # "EfficientSU2": su2_results,
}

df = pd.concat(
    [pd.DataFrame(r) for r in all_results.values() if r],
    ignore_index=True,
)
df["qubits"] = df["problem_index"].map(lambda i: problems[i - 1][0].num_qubits)
df["gap"] = df["energy"] - df["exact_energy"]          # optimality gap (0 = found the exact optimum)
df["hit_optimum"] = df["gap"].abs() < 1e-3

summary = (
    df.groupby(["ansatz_type", "problem_index", "qubits"])
      .agg(energy_mean=("energy", "mean"), energy_std=("energy", "std"),
           exact=("exact_energy", "first"),
           gap_mean=("gap", "mean"), gap_std=("gap", "std"),
           time_mean=("time", "mean"), time_std=("time", "std"),
           hit_rate=("hit_optimum", "mean"))
      .reset_index()
      .fillna({"energy_std": 0, "gap_std": 0, "time_std": 0})
)
display(summary.round(4))

ansatz_types = list(summary["ansatz_type"].unique())
colors = dict(zip(ansatz_types, plt.cm.tab10.colors))
problem_ids = sorted(summary["problem_index"].unique())
xlabels = [f"P{p}\n({summary.loc[summary.problem_index == p, 'qubits'].iloc[0]}q)" for p in problem_ids]
x = np.arange(len(problem_ids))
width = 0.8 / max(len(ansatz_types), 1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: final energy vs exact optimum
ax = axes[0]
for k, a in enumerate(ansatz_types):
    s = summary[summary.ansatz_type == a].set_index("problem_index").reindex(problem_ids)
    pos = x + (k - (len(ansatz_types) - 1) / 2) * width
    ax.bar(pos, s["energy_mean"], width, yerr=s["energy_std"], capsize=3,
           color=colors[a], alpha=0.8, label=a)
    t = df[df.ansatz_type == a]
    ax.scatter(t["problem_index"].map(lambda p: pos[problem_ids.index(p)]), t["energy"],
               color="k", s=12, zorder=3)
ex = summary.groupby("problem_index")["exact"].first().reindex(problem_ids)
ax.scatter(x, ex, marker="D", color="red", s=45, zorder=4, label="Exact optimum")
ax.set_xticks(x); ax.set_xticklabels(xlabels)
ax.set_yscale("log")
ax.set_ylabel("Final energy (log scale)")
ax.set_title("Final energy per problem (dots = individual trials)")
ax.legend()

# Right: optimality gap
ax = axes[1]
for k, a in enumerate(ansatz_types):
    s = summary[summary.ansatz_type == a].set_index("problem_index").reindex(problem_ids)
    pos = x + (k - (len(ansatz_types) - 1) / 2) * width
    ax.bar(pos, s["gap_mean"], width, yerr=s["gap_std"], capsize=3,
           color=colors[a], alpha=0.8, label=a)
    t = df[df.ansatz_type == a]
    ax.scatter(t["problem_index"].map(lambda p: pos[problem_ids.index(p)]), t["gap"],
               color="k", s=12, zorder=3)
ax.axhline(0, color="red", lw=1, ls="--")
ax.set_xticks(x); ax.set_xticklabels(xlabels)
ax.set_ylabel("Energy - exact energy")
ax.set_title("Optimality gap (lower is better, 0 = exact)")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: time per problem
ax = axes[0]
for k, a in enumerate(ansatz_types):
    s = summary[summary.ansatz_type == a].set_index("problem_index").reindex(problem_ids)
    pos = x + (k - (len(ansatz_types) - 1) / 2) * width
    ax.bar(pos, s["time_mean"], width, yerr=s["time_std"], capsize=3,
           color=colors[a], alpha=0.8, label=a)
ax.set_xticks(x); ax.set_xticklabels(xlabels)
ax.set_yscale("log")
ax.set_ylabel("Build + optimization time (s, log scale)")
ax.set_title("Runtime per problem")
ax.legend()

# Right: time scaling with qubit count
ax = axes[1]
for a in ansatz_types:
    t = df[df.ansatz_type == a]
    ax.scatter(t["qubits"], t["time"], color=colors[a], alpha=0.5, s=25)
    g = t.groupby("qubits")["time"].mean()
    ax.plot(g.index, g.values, "-o", color=colors[a], label=a)
ax.set_yscale("log")
ax.set_xlabel("Number of qubits")
ax.set_ylabel("Time (s, log scale)")
ax.set_title("Runtime scaling with problem size")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
markers = dict(zip(ansatz_types, ["o", "s", "^", "v", "D", "P", "X"]))
cmap = plt.cm.viridis
norm = plt.Normalize(df["qubits"].min(), df["qubits"].max())

# Left: every trial, gap vs time
ax = axes[0]
for a in ansatz_types:
    t = df[df.ansatz_type == a]
    sc = ax.scatter(t["time"], t["gap"], c=t["qubits"], cmap=cmap, norm=norm,
                    marker=markers[a], s=60, edgecolor="k", alpha=0.85)
ax.set_xscale("log")
ax.set_yscale("symlog", linthresh=1e-2)
ax.axhline(0, color="red", lw=1, ls="--")
ax.set_xlabel("Time (s, log scale)")
ax.set_ylabel("Optimality gap (symlog)")
ax.set_title("Energy vs time tradeoff, every trial (bottom-left is best)")
fig.colorbar(sc, ax=ax, label="Qubits")
ax.legend(handles=[Line2D([], [], marker=markers[a], ls="", color="gray", mec="k", label=a)
                   for a in ansatz_types])

# Right: per-problem averages with error bars, labelled
ax = axes[1]
for a in ansatz_types:
    s = summary[summary.ansatz_type == a]
    ax.errorbar(s["time_mean"], s["gap_mean"], xerr=s["time_std"], yerr=s["gap_std"],
                fmt="none", ecolor="gray", alpha=0.6, capsize=2)
    ax.scatter(s["time_mean"], s["gap_mean"], c=s["qubits"], cmap=cmap, norm=norm,
               marker=markers[a], s=90, edgecolor="k", zorder=3, label=a)
    for _, r in s.iterrows():
        ax.annotate(f"P{int(r.problem_index)}", (r.time_mean, r.gap_mean),
                    textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xscale("log")
ax.set_yscale("symlog", linthresh=1e-2)
ax.axhline(0, color="red", lw=1, ls="--")
ax.set_xlabel("Mean time (s, log scale)")
ax.set_ylabel("Mean optimality gap (symlog)")
ax.set_title("Energy vs time tradeoff, per-problem mean +/- std")
ax.legend()

plt.tight_layout()
plt.show()